# Pré-treino contrastivo (V-LIBRASIL + MALTA) — o GCN aprende invariância a sinalizante antes do MINDS?

**O que este notebook testa, e o que ele NÃO faz.** A config de entrega —
`ST-GCN --ossos --com-z --z-recentrado` — já mede 94,6%/94,9% de LOSO **sem
pré-treino nenhum**, treinada do zero no MINDS
(`notebook_gcn_variantes.ipynb`). Este notebook testa se inicializar essa mesma
arquitetura com um backbone pré-treinado em V-LIBRASIL+MALTA (contrastivo,
ensinando invariância a sinalizante) melhora o número.

**Não roda `--final` aqui.** É o teste que decide se vale a pena mudar a
config de entrega — só depois de comparar contra o baseline conhecido é que
faz sentido gerar o checkpoint final com (ou sem) esse pré-treino.

## Por que a comparação é justa

A rodada de fine-tuning usa a **mesma semente (20260916)** do baseline sem
pré-treino, os **mesmos hiperparâmetros**, e muda só uma coisa:
`--inicializar <backbone do pré-treino>`. Isso é o que o projeto já exige para
qualquer comparação de variante (ver `docs/protocolo-treinamento.md` §"Regras
que evitam número falso") — sem semente pareada, a diferença entre duas
execuções da MESMA config já chega a ~1,7 pp, maior que muitos efeitos reais.

**Isto ainda não é prova livre de viés.** Se o pré-treino vencer aqui, o
próprio histórico do projeto recomenda confirmar rerodando controle E
candidato numa semente NOVA antes de comprometer a decisão de arquitetura —
só o vencedor sozinho testaria sensibilidade à semente, não superioridade.

## Antes de começar

1. Ative a GPU no runtime. O notebook aborta sem CUDA.
2. Prepare **três pacotes privados**, preservando nomes e estrutura:
   - `landmarks-minds.tar.gz` → pasta `landmarks/`: 800 `.npy` MINDS (`pessoaM*`).
   - `landmarks-vlibrasil.tar.gz` → pasta `landmarks-pretreino-auditado/`: o
     corpus V-LIBRASIL **auditado** atual (4.025 amostras, ver
     `docs/auditoria-pretreino-2026-09-10.md`) — não o pacote antigo
     pré-auditoria. Cada `.npy` com seu sidecar `*.npy.proveniencia.json`.
   - `landmarks-malta.tar.gz` → pasta `landmarks-malta/`: os 9.398 `.npy`
     MALTA (`pessoaT*`), cada um com seu sidecar. É o corpus completo, com
     a UFSC já incluída (`feat/ingest-ufsc`).
3. No Colab, envie os três ao runtime privado; no Kaggle, anexe como datasets
   privados.

**Licença e privacidade:** MINDS é MIT; V-LIBRASIL é **CC BY-NC-ND**
(não-comercial, sem derivações); MALTA agrega fontes universitárias sob
enquadramento de pesquisa, com a UFSC pendente de autorização expressa do NALS
(ver `docs/decisao-datasets-e-licencas.md`). A extração de landmarks não
elimina essas restrições. **Não publique** nada gerado aqui — vídeos,
landmarks, sidecars, pacotes ou checkpoints — em lugar público.

**Limite da avaliação:** os 30 clipes V-LIBRASIL reservados não medem domínio
nem pessoas inéditas (V03 participa da validação interna do pré-treino).
MALTA não sustenta *leave-one-signer-out* (nenhuma palavra passa de 3 pessoas
reais, descontado o fallback `TUFS`) — por isso ele entra só no pré-treino,
nunca na avaliação, que continua sendo o MINDS isolado.

## 1. Ambiente e código

In [ ]:
import os, pathlib, shutil, subprocess, sys

EM_KAGGLE = os.path.exists("/kaggle/working") or "KAGGLE_KERNEL_RUN_TYPE" in os.environ
EM_COLAB = not EM_KAGGLE and ("google.colab" in sys.modules or os.path.exists("/content"))
BASE = pathlib.Path("/kaggle/working" if EM_KAGGLE else "/content" if EM_COLAB else ".")
print("ambiente:", "Kaggle" if EM_KAGGLE else "Colab" if EM_COLAB else "local", "| base:", BASE)

URL = "https://github.com/Heitorvazeg/libras-livre-ai-glasses-brasil.git"
# Branch com as flags de representação no pré-treino (--ossos/--com-z/--z-recentrado).
# Trocar para "dev" quando o PR desta branch for mesclado.
BRANCH = "feat/pretreino-gcn-representacao"
REPO = BASE / "libras-livre-ai-glasses-brasil"
TREINO = REPO / "computer-vision-model" / "treino"

def git(*args, repo=None):
    cmd = ["git"] + (["-C", str(repo)] if repo else []) + list(args)
    return subprocess.run(cmd, capture_output=True, text=True)

if REPO.exists() and (git("rev-parse", "--abbrev-ref", "HEAD", repo=REPO).stdout.strip() != BRANCH
                      or not TREINO.is_dir()):
    print("clone existente está na branch errada ou incompleto — refazendo")
    shutil.rmtree(REPO)

if REPO.exists():
    antes = git("rev-parse", "--short", "HEAD", repo=REPO).stdout.strip()
    git("fetch", "--depth", "1", "origin", BRANCH, repo=REPO)
    git("reset", "--hard", f"origin/{BRANCH}", repo=REPO)
    depois = git("rev-parse", "--short", "HEAD", repo=REPO).stdout.strip()
    print(f"repo atualizado: {antes} -> {depois}" if antes != depois else f"repo já estava atual ({depois})")
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, URL, str(REPO)], check=True)

assert TREINO.is_dir(), f"{TREINO} não existe mesmo após o clone"
print("branch:", git("rev-parse", "--abbrev-ref", "HEAD", repo=REPO).stdout.strip())
print("último commit:", git("log", "-1", "--pretty=%h %s", repo=REPO).stdout.strip())
print("código em:", TREINO)

In [ ]:
import torch
print("torch", torch.__version__, "| CUDA disponível:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Ative a GPU antes de continuar; não executar treino pesado em CPU.")
print("GPU:", torch.cuda.get_device_name(0))

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "scipy"], check=True)

## 2. Três pacotes privados de landmarks

MINDS vai para `PoC/data/landmarks/`; V-LIBRASIL auditado, para
`PoC/data/landmarks-pretreino-auditado/`; MALTA, para
`PoC/data/landmarks-malta/`. Cada um passado explicitamente em `--corpus` no
pré-treino. A extração exige `tarfile` com `filter="data"`, sem fallback
inseguro — só arquivos regulares `.npy`/sidecar, sem duplicata, sem caminho
fora da pasta esperada. Destino já preenchido é erro, para não misturar
pacotes com dado de execução anterior.

In [ ]:
import json
import tarfile
import tempfile

DESTINO = (REPO / "computer-vision-model" / "PoC" / "data").resolve()
DESTINO.mkdir(parents=True, exist_ok=True)
# nome do pacote -> (pasta de destino, prefixo de pessoa esperado)
PACOTES = {
    "landmarks-minds.tar.gz":     ("landmarks",                    "pessoaM"),
    "landmarks-vlibrasil.tar.gz": ("landmarks-pretreino-auditado",  "pessoaV"),
    "landmarks-malta.tar.gz":     ("landmarks-malta",               "pessoaT"),
}

if EM_COLAB:
    from google.colab import files
    print("Selecione os TRÊS pacotes privados: " + ", ".join(PACOTES))
    enviados = files.upload()
    if set(enviados) != set(PACOTES):
        raise ValueError("Envie exatamente os três pacotes esperados, com os nomes indicados.")
    origens = {nome: pathlib.Path(nome).resolve() for nome in PACOTES}
    del enviados
else:
    if EM_KAGGLE:
        # PROCURA em vez de exigir caminho exato — o Kaggle deriva o slug do
        # TÍTULO do dataset, e exigir um slug fixo falha só depois da GPU alocada.
        raiz = pathlib.Path("/kaggle/input")
        achados = {n: sorted(raiz.glob(f"*/{n}")) + sorted(raiz.glob(f"*/*/{n}")) for n in PACOTES}
        faltando = [n for n, v in achados.items() if not v]
        if faltando:
            inventario = {d.name: [q.name for q in sorted(d.iterdir())[:8]]
                          for d in sorted(raiz.iterdir())} if raiz.is_dir() else {}
            raise FileNotFoundError(f"Não achei {faltando} em /kaggle/input.\nEncontrado: {inventario}")
        origens = {n: v[0] for n, v in achados.items()}
        print("pacotes localizados:", {n: str(v) for n, v in origens.items()})
    else:
        raiz_pacotes = pathlib.Path("~").expanduser()
        origens = {nome: raiz_pacotes / nome for nome in PACOTES}

if not hasattr(tarfile, "data_filter"):
    raise RuntimeError("Atualize o Python: extração exige filter='data', sem fallback.")

for nome, (pasta, prefixo) in PACOTES.items():
    if not origens[nome].is_file():
        raise FileNotFoundError(origens[nome])
    alvo = DESTINO / pasta
    if alvo.is_symlink() or (alvo.exists() and (not alvo.is_dir() or any(
        p.name != ".gitkeep" or not p.is_file() or p.is_symlink() for p in alvo.iterdir()
    ))):
        raise RuntimeError(f"Destino já preenchido: {alvo}. Use um runtime/diretório limpo.")

with tempfile.TemporaryDirectory(dir=DESTINO) as temporario:
    staging = pathlib.Path(temporario)
    for nome, (pasta, prefixo) in PACOTES.items():
        with tarfile.open(origens[nome], "r:gz") as tar:
            membros = tar.getmembers()
            vistos = set()
            for membro in membros:
                caminho = pathlib.PurePosixPath(membro.name)
                if caminho.is_absolute() or ".." in caminho.parts or not caminho.parts or caminho.parts[0] != pasta:
                    raise ValueError(f"Caminho inesperado no pacote {nome}: {membro.name}")
                if caminho in vistos:
                    raise ValueError(f"Membro duplicado: {membro.name}")
                vistos.add(caminho)
                if membro.isdir() and len(caminho.parts) == 1:
                    continue
                if not (membro.isfile() and len(caminho.parts) == 2
                        and caminho.name.endswith((".npy", ".npy.proveniencia.json"))):
                    raise ValueError(f"Só landmarks/sidecars regulares são permitidos: {membro.name}")
            tar.extractall(staging, members=membros, filter="data")

        npys = sorted((staging / pasta).glob("*.npy"))
        if not npys or any(not p.name.startswith(prefixo) for p in npys):
            raise ValueError(f"Pacote vazio ou fonte incorreta: {nome} (esperado prefixo {prefixo})")
        for npy in npys:
            sidecar = npy.with_name(npy.name + ".proveniencia.json")
            if pasta != "landmarks" or sidecar.exists():
                if not sidecar.is_file():
                    raise FileNotFoundError(f"Proveniência obrigatória ausente: {sidecar.name}")
                meta = json.loads(sidecar.read_text(encoding="utf-8"))
                if not isinstance(meta, dict) or not meta:
                    raise ValueError(f"Sidecar deve conter um objeto JSON não vazio: {sidecar.name}")
        for sidecar in (staging / pasta).glob("*.npy.proveniencia.json"):
            if not sidecar.with_name(sidecar.name.removesuffix(".proveniencia.json")).is_file():
                raise ValueError(f"Sidecar sem landmark correspondente: {sidecar.name}")
        pessoas = sorted({p.name.split("_")[0] for p in npys})
        if pasta == "landmarks-pretreino-auditado" and "pessoaV03" not in pessoas:
            raise ValueError("V03 não está no corpus; não é possível reservar a validação explícita.")
        print(f"{pasta}: {len(npys)} clipes | {len(pessoas)} pessoa(s)")

    for pasta, _ in PACOTES.values():
        alvo = DESTINO / pasta
        alvo.mkdir(exist_ok=True)
        for item in (staging / pasta).iterdir():
            shutil.move(str(item), str(alvo / item.name))

CORPUS_VLIBRASIL = DESTINO / "landmarks-pretreino-auditado"
CORPUS_MALTA = DESTINO / "landmarks-malta"
print("prontos:", CORPUS_VLIBRASIL, "|", CORPUS_MALTA)

## 3. Auditoria → pré-treino contrastivo (V-LIBRASIL + MALTA)

`--fontes vlibrasil,malta` e a representação vencedora (`--ossos --com-z
--z-recentrado`) — a MESMA que o fine-tuning vai usar. Isso importa: o
backbone salvo carrega no fine-tuning por CONTAGEM de tensores/shapes; se a
representação divergir entre os dois estágios (por exemplo, pré-treinar com
`--com-z` mas sem `--z-recentrado`), os shapes ainda batem e o carregamento
"funciona", mas os pesos da primeira camada aprenderam um referencial de z
diferente do que vão receber — silencioso. Use sempre a MESMA lista de flags
de representação nas duas chamadas abaixo.

`--auditar` confere isolamento e proveniência antes de construir qualquer
modelo; `selftest.py` valida o encanamento sintético depois. Nenhum dos dois
é opcional.

In [ ]:
from datetime import datetime
from uuid import uuid4

TREINO = TREINO.resolve()
EXP = BASE.resolve() / "experimentos-privados" / (
    "pretreino-malta-" + datetime.now().strftime("%Y%m%d-%H%M%S") + "-" + uuid4().hex[:8]
)
EXP.mkdir(parents=True, exist_ok=False)
SAIDA_PRE = EXP / "pretreino-vlibrasil-malta-contrastivo"
SAIDA_FT = EXP / "finetuning-minds-loso-com-pretreino"
CHECKPOINT = SAIDA_PRE / "backbone_gcn.pt"

# A representação: reaparece idêntica no pré-treino e no fine-tuning (célula seguinte).
REPRESENTACAO = ["--ossos", "--com-z", "--z-recentrado"]

PRE_ARGS = [
    "--corpus", str(CORPUS_VLIBRASIL), "--corpus", str(CORPUS_MALTA),
    "--fontes", "vlibrasil,malta",
    "--arquitetura", "gcn", *REPRESENTACAO,
    "--objetivo", "contrastivo", "--pessoa-val", "V03",
    "--epocas", "15", "--lr", "1e-4", "--batch", "64",
    "--p-classes", "32", "--k-exemplos", "2", "--semente", "0",
    "--saida", str(SAIDA_PRE),
]

# Obrigatório: audita e sai antes de construir qualquer modelo.
subprocess.run([sys.executable, "pretreinar.py", *PRE_ARGS, "--dispositivo", "cpu", "--auditar"],
              cwd=TREINO, check=True)
subprocess.run([sys.executable, "selftest.py"], cwd=TREINO, check=True)
print("Saídas privadas desta execução:", EXP)

In [ ]:
# Estágio 1: pré-treino contrastivo V-LIBRASIL+MALTA, com a representação vencedora.
# MINDS nunca entra aqui — é o conjunto de avaliação (a auditoria acima já barra isso).
subprocess.run([sys.executable, "pretreinar.py", *PRE_ARGS, "--dispositivo", "cuda"],
              cwd=TREINO, check=True)
if not CHECKPOINT.is_file() or not CHECKPOINT.with_suffix(".json").is_file():
    raise FileNotFoundError("Pré-treino não produziu backbone_gcn.pt e seu JSON.")
print("Backbone que será transferido:", CHECKPOINT)

## 4. Fine-tuning LOSO no MINDS — a comparação que decide

**Mesma semente do baseline sem pré-treino (`20260916`), mesmos
hiperparâmetros, mesma representação.** A única diferença desta chamada para
a que produziu 94,6% é `--inicializar`. `--folds 0` roda as 8 rodadas
completas; `treinar.py` retoma sozinho de `rodadas/NN-<pessoa>.json` se a
sessão cair no meio.

**Não há `--final` aqui.** Este é o número que decide, não a entrega.

In [ ]:
SEMENTE_BASELINE = "20260916"  # a mesma do G-ossos-z medido em notebook_gcn_variantes.ipynb

FT_ARGS = [
    "--arquitetura", "gcn", "--fontes", "minds", *REPRESENTACAO,
    "--dispositivo", "cuda", "--epocas", "120", "--lr", "1e-3", "--batch", "64",
    "--agendador", "cosseno", "--folds", "0", "--semente", SEMENTE_BASELINE,
    "--inicializar", str(CHECKPOINT), "--saida", str(SAIDA_FT),
]
if not CHECKPOINT.is_file():
    raise FileNotFoundError(CHECKPOINT)
subprocess.run([sys.executable, "treinar.py", *FT_ARGS], cwd=TREINO, check=True)

## 5. O número, contra o baseline conhecido

In [ ]:
import re

BASELINE_PP = {"semente 20260916 (sem pré-treino)": 94.6, "semente confirmatória (sem pré-treino)": 94.9}

texto = (SAIDA_FT / "relatorio.md").read_text(encoding="utf-8")
folds = {k: float(v) for k, v in re.findall(r"^\| (M\d+) \| ([\d.]+)%", texto, re.M)}
media = float(re.search(r"média = ([\d.]+)%", texto).group(1))

print(f"{'rodada':<10}{'acurácia':>10}")
for pessoa, acc in sorted(folds.items()):
    print(f"{pessoa:<10}{acc:>9.1f}%")
print(f"\n{'MÉDIA (com pré-treino)':<28}{media:>6.1f}%")
for rot, base in BASELINE_PP.items():
    print(f"{rot:<28}{base:>6.1f}%  | diferença: {media - base:+.1f} pp")

print("""
LEIA ANTES DE CONCLUIR:
  - variação de ~1,7 pp entre execuções da MESMA config já foi medida neste
    projeto. Diferença menor que isso é ruído, não efeito do pré-treino.
  - olhe a CONSISTÊNCIA por rodada (quantas subiram, quantas desceram), não
    só a média — foi o que sustentou a decisão da imputação (7 de 8 folds).
  - se isto vencer, ainda não é decisão: confirmar exige rerodar CONTROLE E
    CANDIDATO numa semente NOVA (docs/protocolo-treinamento.md §7).
""")

## 6. Baixar todos os artefatos — uso privado

Inclui backbone `.pt`, JSON de metadados, relatório e matriz de confusão de
cada rodada. Landmarks e pacotes de entrada ficam fora.

In [ ]:
for rel in sorted(EXP.rglob("relatorio.md")):
    print("=" * 70, "\n", rel.relative_to(EXP))
    print(rel.read_text(encoding="utf-8")[:1500])

artefatos = sorted(p for p in EXP.rglob("*") if p.is_file())
if not artefatos:
    raise RuntimeError("Nenhum artefato para baixar.")
for artefato in artefatos:
    print(artefato.relative_to(EXP), "|", artefato.stat().st_size, "bytes")

arquivo = EXP.parent / f"{EXP.name}.tar.gz"
with tarfile.open(arquivo, "w:gz") as tar:
    tar.add(EXP, arcname=EXP.name)
print("\narquivo pronto:", arquivo, "|", arquivo.stat().st_size, "bytes")

if EM_COLAB:
    from google.colab import files
    files.download(str(arquivo))
else:
    print("Kaggle/local: baixe manualmente do painel de saída (privado).")